<a href="https://colab.research.google.com/github/Progammingfrenzy/diabetes-ml-classification/blob/main/Diabetes_FT_Transformer_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import torch
import sklearn

print("Pandas:", pd.__version__)
print("PyTorch:", torch.__version__)
print("Scikit-learn", sklearn.__version__)

Pandas: 2.2.3
PyTorch: 2.14.1+cpu
Scikit-learn 1.6.1


In [2]:
import os

#Show files currently uploaded to Colab
print(os.listdir("/content"))

['.config', 'sample_data']


In [3]:
# Load dataset directly from GitHub
url = "https://raw.githubusercontent.com/razibdash/Diabetes-Health-Indicators/main/diabetes_012_health_indicators_BRFSS2015.csv"

df = pd.read_csv(url)

print("Dataset shape:", df.shape)


Dataset shape: (253680, 22)


In [4]:
# Check for Missing Values
print("Missing values:", df.isnull().sum().sum())

# Check for duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Display data types
print(df.dtypes)

Missing values: 0
Duplicate rows: 23899
Diabetes_012            float64
HighBP                  float64
HighChol                float64
CholCheck               float64
BMI                     float64
Smoker                  float64
Stroke                  float64
HeartDiseaseorAttack    float64
PhysActivity            float64
Fruits                  float64
Veggies                 float64
HvyAlcoholConsump       float64
AnyHealthcare           float64
NoDocbcCost             float64
GenHlth                 float64
MentHlth                float64
PhysHlth                float64
DiffWalk                float64
Sex                     float64
Age                     float64
Education               float64
Income                  float64
dtype: object


In [5]:

# Binary features (0 or 1)
binary_cols = [
    "HighBP", "HighChol", "CholCheck",
    "Smoker", "Stroke", "HeartDiseaseorAttack",
    "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare",
    "NoDocbcCost", "DiffWalk", "Sex"
]

# Ordinal features (ordered categories)
ordinal_cols = [
    "GenHlth", "Age", "Education", "Income"
]

# Numerical features
numerical_cols = [
    "BMI", "MentHlth", "PhysHlth"
]


# Define our input features and target
X = df.drop(columns=["Diabetes_012"])
y = df["Diabetes_012"].astype(int)

print("Features:", X.shape)
print("Target:", y.shape)

# Check that all 21 features are accounted for
all_cols = binary_cols + ordinal_cols + numerical_cols

assert len(all_cols) == len(set(all_cols))
assert set(all_cols) == set(X.columns)

print("Binary features:", len(binary_cols))
print("Ordinal features:", len(ordinal_cols))
print("Numerical features:", len(numerical_cols))
print("Total features:", len(all_cols))



Features: (253680, 21)
Target: (253680,)
Binary features: 14
Ordinal features: 4
Numerical features: 3
Total features: 21


In [6]:

from sklearn.model_selection import train_test_split

# Split into 80% training and 20% temporary data
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Divide the temporary data into validation and test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)


Training: (202944, 21)
Validation: (25368, 21)
Testing: (25368, 21)


In [7]:

# Show percentage of each diabetes class in training
print(y_train.value_counts(normalize=True) * 100)

Diabetes_012
0    84.240973
2    13.933400
1     1.825627
Name: proportion, dtype: float64


In [8]:

# Check validation class percentages
print(y_val.value_counts(normalize=True) * 100)

# Check testing class percentages
print(y_test.value_counts(normalize=True) * 100)


Diabetes_012
0    84.239987
2    13.934879
1     1.825134
Name: proportion, dtype: float64
Diabetes_012
0    84.243929
2    13.930937
1     1.825134
Name: proportion, dtype: float64


In [9]:

# Identify matching feature combinations
train_keys = pd.util.hash_pandas_object(X_train, index=False)
test_keys = pd.util.hash_pandas_object(X_test, index=False)

# Count matching test records
overlap = test_keys.isin(set(train_keys)).sum()

print("Overlapping test records:", overlap)


Overlapping test records: 3480


In [10]:

# Check whether our previous work is still loaded
print("Dataset loaded:", "df" in globals())
print("Training data ready:", "X_train" in globals())


Dataset loaded: True
Training data ready: True


In [11]:

from sklearn.preprocessing import StandardScaler

# Create a scaler
scaler = StandardScaler()

# Make copies of our datasets
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

# Scale numerical features
X_train_scaled[numerical_cols] = scaler.fit_transform(
    X_train[numerical_cols]
)

X_val_scaled[numerical_cols] = scaler.transform(
    X_val[numerical_cols]
)

X_test_scaled[numerical_cols] = scaler.transform(
    X_test[numerical_cols]
)

# View the scaled training features
display(X_train_scaled[numerical_cols].head())


,BMI,MentHlth,PhysHlth
185494,-0.512552,-0.429290,-0.485961
217030,1.301115,-0.429290,-0.485961
641,0.998837,-0.429290,-0.485961
217863,-0.210274,-0.429290,0.088138
53677,-0.059135,-0.023967,-0.485961


In [12]:
# Check the mean and standard deviation
print(X_train_scaled[numerical_cols].mean().round(2))
print(X_train_scaled[numerical_cols].std().round(2))


BMI        -0.0
MentHlth    0.0
PhysHlth    0.0
dtype: float64
BMI         1.0
MentHlth    1.0
PhysHlth    1.0
dtype: float64


In [13]:
# Combine binary and ordinal feature names
categorical_cols = binary_cols + ordinal_cols

# Check the feature counts
print("Categorical features:", len(categorical_cols))
print("Numerical features:", len(numerical_cols))


Categorical features: 18
Numerical features: 3


In [14]:

# Examine categorical values in the training dataset
category_summary = X_train[categorical_cols].agg(
    ["min", "max", "nunique"]
).T

display(category_summary)


,min,max,nunique
HighBP,0.0,1.0,2.0
HighChol,0.0,1.0,2.0
CholCheck,0.0,1.0,2.0
Smoker,0.0,1.0,2.0
Stroke,0.0,1.0,2.0
HeartDiseaseorAttack,0.0,1.0,2.0
PhysActivity,0.0,1.0,2.0
Fruits,0.0,1.0,2.0
Veggies,0.0,1.0,2.0
HvyAlcoholConsump,0.0,1.0,2.0


In [15]:

# Convert categorical training features to integers
X_train_cat = X_train[categorical_cols].astype(int).copy()

# Make ordinal categories start at zero
X_train_cat[ordinal_cols] -= 1

# Display converted features
display(X_train_cat[["Smoker", "Age", "GenHlth"]].head())


,Smoker,Age,GenHlth
185494,0,9,1
217030,1,0,0
641,0,10,1
217863,0,2,1
53677,1,8,2


In [16]:

# Convert validation and testing features to integers
X_val_cat = X_val[categorical_cols].astype(int).copy()
X_test_cat = X_test[categorical_cols].astype(int).copy()

# Make ordinal categories start at zero
X_val_cat[ordinal_cols] -= 1
X_test_cat[ordinal_cols] -= 1

# Verify dimensions
print("Training:", X_train_cat.shape)
print("Validation:", X_val_cat.shape)
print("Testing:", X_test_cat.shape)


Training: (202944, 18)
Validation: (25368, 18)
Testing: (25368, 18)


In [17]:

# Determine the number of categories per feature
cardinalities = X_train_cat.nunique().tolist()

print("Category cardinalities:", cardinalities)
print("Total categorical features:", len(cardinalities))


Category cardinalities: [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 5, 13, 6, 8]
Total categorical features: 18


In [18]:

# Verify categorical indices in all datasets
for data in [X_train_cat, X_val_cat, X_test_cat]:
    for col, size in zip(categorical_cols, cardinalities):
        assert data[col].min() >= 0
        assert data[col].max() < size

print("All categorical indices are valid!")


All categorical indices are valid!


In [19]:

# Convert numerical features
train_num = torch.tensor(
    X_train_scaled[numerical_cols].values,
    dtype=torch.float32
)

# Convert categorical features
train_cat = torch.tensor(
    X_train_cat.values,
    dtype=torch.long
)

# Convert target labels
train_labels = torch.tensor(
    y_train.values,
    dtype=torch.long
)

print("Numerical:", train_num.shape)
print("Categorical:", train_cat.shape)
print("Labels:", train_labels.shape)


Numerical: torch.Size([202944, 3])
Categorical: torch.Size([202944, 18])
Labels: torch.Size([202944])


In [20]:

# Convert validation data
val_num = torch.tensor(
    X_val_scaled[numerical_cols].values,
    dtype=torch.float32
)

val_cat = torch.tensor(
    X_val_cat.values,
    dtype=torch.long
)

val_labels = torch.tensor(
    y_val.values,
    dtype=torch.long
)

# Convert testing data
test_num = torch.tensor(
    X_test_scaled[numerical_cols].values,
    dtype=torch.float32
)

test_cat = torch.tensor(
    X_test_cat.values,
    dtype=torch.long
)

test_labels = torch.tensor(
    y_test.values,
    dtype=torch.long
)


In [21]:

print("Validation:", val_num.shape, val_cat.shape, val_labels.shape)
print("Testing:", test_num.shape, test_cat.shape, test_labels.shape)


Validation: torch.Size([25368, 3]) torch.Size([25368, 18]) torch.Size([25368])
Testing: torch.Size([25368, 3]) torch.Size([25368, 18]) torch.Size([25368])


In [22]:

from torch.utils.data import TensorDataset

train_dataset = TensorDataset(train_num, train_cat, train_labels)
val_dataset = TensorDataset(val_num, val_cat, val_labels)
test_dataset = TensorDataset(test_num, test_cat, test_labels)

print("Training:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Testing:", len(test_dataset))


Training: 202944
Validation: 25368
Testing: 25368


In [23]:

from torch.utils.data import DataLoader

batch_size = 256

train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True
)

val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False
)

test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))


Training batches: 793
Validation batches: 100
Testing batches: 100


In [24]:

import torch.nn as nn

class FTTransformer(nn.Module):
    def __init__(self, n_num, cardinalities,
                 d_model=32, n_heads=4,
                 n_layers=2, dropout=0.1):
        super().__init__()

        # Numerical feature tokenizer
        self.num_weight = nn.Parameter(
            torch.randn(n_num, d_model) * 0.02
        )
        self.num_bias = nn.Parameter(
            torch.zeros(n_num, d_model)
        )

        # Categorical feature embeddings
        self.cat_embeddings = nn.ModuleList([
            nn.Embedding(size, d_model)
            for size in cardinalities
        ])

        # Classification token
        self.cls_token = nn.Parameter(
            torch.zeros(1, 1, d_model)
        )

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=n_layers,
            enable_nested_tensor=False
        )

        # Classification head
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.ReLU(),
            nn.Linear(d_model, 3)
        )

    def forward(self, x_num, x_cat):
        # Create numerical tokens
        num_tokens = (
            x_num.unsqueeze(-1) * self.num_weight
            + self.num_bias
        )

        # Create categorical tokens
        cat_tokens = torch.stack([
            emb(x_cat[:, i])
            for i, emb in enumerate(self.cat_embeddings)
        ], dim=1)

        # Combine all feature tokens with CLS token
        cls = self.cls_token.expand(x_num.size(0), -1, -1)

        tokens = torch.cat(
            [cls, num_tokens, cat_tokens], dim=1
        )

        # Process features through attention layers
        output = self.transformer(tokens)

        # Produce three class scores
        return self.classifier(output[:, 0])
